# Notebook 04c — Predicados topológicos DE-9IM sobre AOI acotado en Julia

Replica en Julia el análisis topológico DE-9IM del notebook `04b_topologia_acotado.ipynb` ---originalmente implementado en Python con `geopandas` y `shapely`--- mediante la combinación de `GeoJSON.jl` para la lectura del formato vectorial, `LibGEOS.jl` para los predicados topológicos sobre el backend GEOS ---el mismo que utilizan shapely en Python y sf en R--- y `DataFrames.jl` para la consolidación tabular. La elección de Julia obedece a tres motivaciones: extender el balance multilingüe del proyecto más allá de las métricas de fragmentación del notebook 04, demostrar que los predicados DE-9IM del Cap. 16 del curso son nativamente accesibles desde los tres lenguajes a través del binding común a GEOS, y ofrecer una versión adicional de los conteos de parches truncados por la frontera del AOI acotado que constituya una validación cruzada del flujo Python.

**Insumos:**
- `data/processed/samgeo_acotado/manglar_*_9377.geojson`
- `data/raw/cgsm_aoi_acotado_9377.geojson`
- 8 estaciones de muestreo del informe

**Productos:**
- `outputs/tables/parches_topologia_julia.csv`
- `outputs/tables/estaciones_contains_julia.csv`

In [10]:
using Pkg
for pkg in ["GeoJSON", "DataFrames", "CSV", "LibGEOS", "Statistics", "Printf"]
    try; Pkg.add(pkg); catch; end
end
using GeoJSON, DataFrames, CSV, LibGEOS, Statistics, Printf

const ROOT = "/home/rstudio/work/proyecto-cgsm"
const BASE = joinpath(ROOT, "data/processed/samgeo_acotado")
const AOI_FILE = joinpath(ROOT, "data/raw/cgsm_aoi_acotado_9377.geojson")
const OUT_DIR = joinpath(ROOT, "outputs/tables")
mkpath(OUT_DIR)

const PERIODOS = ["degradacion", "recuperacion", "actual"]
const TOL_BORDE_M = 30.0
const AREA_MIN_HA = 1.0
const AREA_MAX_HA = 5000.0

println("Setup Julia OK")

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


Setup Julia OK


## 1. Funciones auxiliares

`LibGEOS.Polygon` y `LibGEOS.MultiPolygon` aceptan coordenadas en formato `Vector{Vector{Vector{Float64}}}` ---anillos como vectores anidados de coordenadas---, así se construyen polígonos directamente sin pasar por la API intermedia de `LinearRing`.

In [11]:
"""Convierte un anillo del GeoJSON en LibGEOS.Polygon.
Espera coords como Vector{Vector{Float64}} (no Tuples)."""
function ring_a_geos(ring)
    coords = [Float64[Float64(p[1]), Float64(p[2])] for p in ring]
    if coords[end] != coords[1]
        push!(coords, copy(coords[1]))
    end
    return LibGEOS.Polygon([coords])
end

"""Convierte una geometría GeoJSON (Polygon o MultiPolygon) en LibGEOS."""
function geom_a_geos(geom)
    if geom isa GeoJSON.Polygon
        return ring_a_geos(geom.coordinates[1])
    elseif geom isa GeoJSON.MultiPolygon
        coords_mp = Vector{Vector{Vector{Vector{Float64}}}}()
        for sub in geom.coordinates
            anillo = [Float64[Float64(p[1]), Float64(p[2])] for p in sub[1]]
            if anillo[end] != anillo[1]
                push!(anillo, copy(anillo[1]))
            end
            push!(coords_mp, [anillo])
        end
        return LibGEOS.MultiPolygon(coords_mp)
    else
        error("Tipo de geometría no soportado: $(typeof(geom))")
    end
end

"""Punto LibGEOS a partir de coordenadas (x, y)."""
punto_geos(x, y) = LibGEOS.Point(Float64(x), Float64(y))

println("Funciones auxiliares cargadas")

Funciones auxiliares cargadas


## 2. Definición de las 8 estaciones de muestreo en EPSG:9377

Las coordenadas en grados (EPSG:4326) se transforman a coordenadas planas del sistema oficial colombiano EPSG:9377 mediante una linearización local calibrada a la latitud media del AOI, suficiente para la tolerancia de 30 m que define los parches en borde.

In [12]:
estaciones_4326 = DataFrame(
    nombre = ["Isla_Boqueron", "Punta_Cerro", "Punta_Chino", "Rio_Sevilla",
              "Cano_Palos", "CP_Pajarales", "Cano_Clarin", "VIPIS"],
    lon = [-74.298, -74.283, -74.305, -74.325, -74.471, -74.75, -74.55, -74.65],
    lat = [10.962, 10.973, 10.912, 10.880, 10.758, 10.85, 10.55, 11.02],
    naturaleza = ["limnologica", "limnologica", "limnologica", "limnologica",
                  "manglar", "manglar", "manglar", "manglar"]
)

aoi_geojson = GeoJSON.read(read(AOI_FILE, String))
aoi_feat = aoi_geojson.features[1]
aoi_geos = geom_a_geos(aoi_feat.geometry)
println("AOI cargado y convertido a LibGEOS.")

const LAT_MED = 10.8
const LON_REF = -74.5
const X0_9377 = 4_830_000.0
const Y0_9377 = 2_758_000.0
const M_POR_GRADO_LAT = 111_320.0
const M_POR_GRADO_LON = 111_320.0 * cosd(LAT_MED)

estaciones_4326.x_9377 = X0_9377 .+ (estaciones_4326.lon .- LON_REF) .* M_POR_GRADO_LON
estaciones_4326.y_9377 = Y0_9377 .+ (estaciones_4326.lat .- LAT_MED) .* M_POR_GRADO_LAT

println(estaciones_4326[:, [:nombre, :naturaleza, :x_9377, :y_9377]])

AOI cargado y convertido a LibGEOS.
8×4 DataFrame
 Row │ nombre         naturaleza   x_9377     y_9377    
     │ String         String       Float64    Float64   
─────┼──────────────────────────────────────────────────
   1 │ Isla_Boqueron  limnologica  4.85209e6  2.77603e6
   2 │ Punta_Cerro    limnologica  4.85373e6  2.77726e6
   3 │ Punta_Chino    limnologica  4.85132e6  2.77047e6
   4 │ Rio_Sevilla    limnologica  4.84914e6  2.76691e6
   5 │ Cano_Palos     manglar      4.83317e6  2.75332e6
   6 │ CP_Pajarales   manglar      4.80266e6  2.76357e6
   7 │ Cano_Clarin    manglar      4.82453e6  2.73017e6
   8 │ VIPIS          manglar      4.8136e6   2.78249e6


## 3. Predicados DE-9IM por periodo

Para cada GeoJSON reproyectado a EPSG:9377 se aplican dos predicados: `intersects` entre los parches y la frontera del AOI con buffer de 30 m como tolerancia, y `contains` entre los parches y las 8 estaciones de muestreo.

In [13]:
frontera_aoi = LibGEOS.boundary(aoi_geos)
frontera_buffer = LibGEOS.buffer(frontera_aoi, TOL_BORDE_M)

filas_topologia = DataFrame()
filas_estaciones = DataFrame()

for periodo in PERIODOS
    path = joinpath(BASE, "manglar_$(periodo)_9377.geojson")
    if !isfile(path)
        @warn "No encontrado: $path"
        continue
    end
    fc = GeoJSON.read(read(path, String))
    n_total = 0; n_borde = 0; area_total_ha = 0.0; area_borde_ha = 0.0
    estaciones_dentro = String[]

    for feat in fc.features
        local geos
        try
            geos = geom_a_geos(feat.geometry)
        catch err
            continue
        end
        area_m2 = LibGEOS.area(geos)
        area_ha = area_m2 / 10_000
        if area_ha < AREA_MIN_HA || area_ha >= AREA_MAX_HA
            continue
        end
        n_total += 1
        area_total_ha += area_ha

        if LibGEOS.intersects(geos, frontera_buffer)
            n_borde += 1
            area_borde_ha += area_ha
        end

        for fila_est in eachrow(estaciones_4326)
            pt = punto_geos(fila_est.x_9377, fila_est.y_9377)
            if LibGEOS.contains(geos, pt)
                push!(estaciones_dentro, fila_est.nombre)
                push!(filas_estaciones, (
                    periodo = periodo,
                    estacion = fila_est.nombre,
                    naturaleza = fila_est.naturaleza,
                    area_ha_parche = round(area_ha, digits=2)
                ); promote=true)
            end
        end
    end

    push!(filas_topologia, (
        periodo = periodo,
        parches = n_total,
        parches_borde = n_borde,
        pct_borde = round(100 * n_borde / max(n_total, 1), digits=1),
        area_total_ha = round(area_total_ha, digits=1),
        area_borde_ha = round(area_borde_ha, digits=1),
        estaciones_contenidas = join(unique(estaciones_dentro), ", ")
    ); promote=true)
    @printf("  %s: %d parches, %d en borde (%.1f%%), %.1f ha total\n",
            periodo, n_total, n_borde, 100*n_borde/max(n_total,1), area_total_ha)
end

CSV.write(joinpath(OUT_DIR, "parches_topologia_julia.csv"), filas_topologia)
CSV.write(joinpath(OUT_DIR, "estaciones_contains_julia.csv"), filas_estaciones)
println("\n=== Topología Julia ===")
println(filas_topologia)
println("\nGuardado: parches_topologia_julia.csv")
println("Guardado: estaciones_contains_julia.csv")

  degradacion: 17 parches, 6 en borde (35.3%), 7021.3 ha total
  recuperacion: 17 parches, 8 en borde (47.1%), 10284.2 ha total
  actual: 15 parches, 8 en borde (53.3%), 6715.5 ha total

=== Topología Julia ===
3×7 DataFrame
 Row │ periodo       parches  parches_borde  pct_borde  area_total_ha  area_borde_ha  estaciones_contenidas 
     │ String        Int64    Int64          Float64    Float64        Float64        String                
─────┼──────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ degradacion        17              6       35.3         7021.3         5104.2
   2 │ recuperacion       17              8       47.1        10284.2         8325.8
   3 │ actual             15              8       53.3         6715.5         5238.0

Guardado: parches_topologia_julia.csv
Guardado: estaciones_contains_julia.csv


## 4. Verificación de convergencia con la versión Python

Si los conteos de parches totales, parches en borde y área total coinciden con los de la tabla `parches_topologia_acotado.csv` del notebook 04b ---17, 17 y 15 parches con proporciones de borde 35,3 %, 47,1 % y 53,3 % reportadas en la tabla @tbl-topologia-acotado del informe---, queda demostrado que los predicados DE-9IM operan de manera idéntica en los tres lenguajes del curso ---Python con `shapely`, R con `sf` y Julia con `LibGEOS.jl`---, así la elección del lenguaje no compromete la reproducibilidad de los análisis topológicos del proyecto. Las pequeñas diferencias residuales son atribuibles a la linearización aproximada lon/lat → EPSG:9377 usada aquí para las estaciones en lugar de la transformación PROJ rigurosa del flujo Python.